# Cartly – Retail Intelligence & Revenue Protection System
**Tagline**: Turn Customer Insights into Revenue Protection

This notebook demonstrates the end-to-end data science and machine learning pipeline powering **Cartly**:
1. Data Ingestion & Canonical Schema Validation
2. Financial Cleaning & Cancellation Treatment
3. Customer Analytics (RFM Segmentation)
4. Operational Churn Snapshot Generation (180-Day History, 90-Day Target)
5. Model Training & Comparison (Logistic Regression vs. Random Forest)
6. Cartly Smart Revenue Protection Engine (Revenue at Risk Baseline Formula)
7. Smart Retention Action Engine

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

from src.data.loader import load_dataset
from src.data.cleaner import clean_retail_data
from src.analytics.executive import calculate_executive_kpis
from src.analytics.rfm import calculate_rfm_metrics, score_rfm, segment_customers
from src.features.churn_features import generate_chronological_churn_data
from src.models.churn_model import train_and_compare_models
from src.business.revenue_protection import calculate_revenue_at_risk, calculate_revenue_protection_summary
from src.business.recommendations import generate_all_recommendations

## 1. Load and Clean Retail Data
Isolate returns/cancellations (Invoice starting with 'C'), remove invalid prices, and deduplicate records.

In [ ]:
raw_df, meta = load_dataset('../data/sample_retail_50k.csv')
print('Raw Data Shape:', raw_df.shape)
cleaned_df, report = clean_retail_data(raw_df, drop_missing_customers=True)
print('Cleaned Rows:', len(cleaned_df))
print(f"Gross Sales: £{report['gross_sales_gbp']:,.2f}")
print(f"Returns: £{report['returns_amount_gbp']:,.2f}")
print(f"Net Revenue: £{report['net_revenue_gbp']:,.2f}")

## 2. Customer RFM Segmentation
Calculate Recency, Frequency, and Monetary spend and assign quintile segments (Champions, Loyal Customers, At Risk, Hibernating).

In [ ]:
rfm_raw = calculate_rfm_metrics(cleaned_df)
scored_rfm = score_rfm(rfm_raw, q=5)
segmented_rfm = segment_customers(scored_rfm)
segmented_rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Segment']].head(10)

## 3. Churn Feature Engineering & Chronological Train/Test Split
Strict 180-day history window and 90-day future prediction window to avoid any data leakage.

In [ ]:
train_df, test_df, split_meta = generate_chronological_churn_data(cleaned_df)
print('Train snapshot instances:', len(train_df))
print('Test snapshot instances:', len(test_df))
print('Split metadata:', split_meta)

comparison = train_and_compare_models(train_df, test_df, save_best_model=True)
print('Best Model:', comparison['best_model_name'])
print('LR F1:', comparison['logistic_regression']['metrics']['f1_score'])
print('RF F1:', comparison['random_forest']['metrics']['f1_score'])

## 4. Smart Revenue Protection Engine
Formula: Estimated Revenue at Risk = Churn Probability × Estimated Future Revenue.

In [ ]:
from src.models.churn_model import predict_customer_churn_probabilities, load_model_bundle
bundle = load_model_bundle()
preds = predict_customer_churn_probabilities(bundle, test_df)
protected_df = calculate_revenue_at_risk(preds)
summary = calculate_revenue_protection_summary(protected_df)
print(f"Total Revenue at Risk: £{summary['total_revenue_at_risk']:,.2f}")
print(f"Immediate Attention Customers: {summary['immediate_attention_customers']}")
protected_df[['risk_rank', 'customer_id', 'priority', 'risk_tier', 'churn_probability', 'estimated_future_revenue', 'revenue_at_risk']].head(10)

## 5. Smart Retention Recommendations
Rule-based tailored retention actions with explicit business justifications.

In [ ]:
recs_df = generate_all_recommendations(protected_df)
recs_df[['customer_id', 'priority', 'risk_level', 'estimated_revenue_at_risk', 'recommended_action']].head(10)